# ⚡️ QuickStart: Scene Index

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/quickstart/Scene%20Index%20QuickStart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This guide shows how to create a visual scene index for a video and retrieve relevant moments with semantic search.

Vision models create timestamped scene descriptions that you can index for visual retrieval.

You can build retrieval experiences for queries like:
![](https://raw.githubusercontent.com/video-db/videodb-cookbook/main/images/scene_index/intro.png)

## Setup
---

### 📦  Installing packages   

In [1]:
!pip install -q videodb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.5/86.5 kB 1.8 MB/s eta 0:00:00


### 🔑 API Keys

In [2]:
import videodb
import os
from getpass import getpass

api_key = getpass("Please enter your VideoDB API Key: ")

os.environ["VIDEO_DB_API_KEY"] = api_key

Please enter your VideoDB API Key: ··········


### 🌐 Connect to VideoDB

In [3]:
from videodb import connect

conn = connect()
coll = conn.get_collection()


### 🎥  Upload Video

In [4]:
video = coll.upload(url="https://www.youtube.com/watch?v=LejnTJL173Y")

print(f"Video ID : {video.id}")
print(f"Title    : {video.name}")
print(f"Duration : {video.length:.0f}s ({video.length/60:.1f} min)")

Video ID : m-z-019f8f57-f740-7d73-86a7-a56cd90c323e
Title    : Best of Rust Cohle True Detective Season 1
Duration : 530s (8.8 min)


In [5]:
video.play()

## 📇 Index Scenes
---

Create timestamped visual descriptions, then build a semantic index from the scene artifact.

In [6]:
print("Analyzing visual scenes... this might take a moment.")

scene_understanding = video.understand(
    segmentation={"type": "time", "seconds": 10},
    analyzers=[
        {
            "type": "vlm",
            "name": "scene",
            "sampling": {
                "strategy": "uniform",
                "frame_count": 1,
            },
            "config": {
                "prompt": "describe the image in 100 words",
                "schema": {
                    "description": "string",
                },
            },
        },
    ],
)
scene_understanding.wait_until_complete()

scene_analyzer = scene_understanding.get_analyzer("scene")


Analyzing visual scenes... this might take a moment.


### Understanding Configuration

This run analyzes 10-second video segments. The VLM samples one representative middle frame of each segment and writes a detailed description for semantic retrieval.

In [7]:
scene_index = video.index(
    name="scene",
    source=scene_analyzer,
    use_for=["semantic"],
    fields={"semantic": ["description"]},
)

scene_index.wait_until_complete()

index_id = scene_index.index_id
print(f"Scene index ready: {scene_index.name}")


Scene index ready: scene


> Semantic search is available after the index status is `ready`.

In [8]:
res = video.semantic_search(
    query="religious gathering",
    index_ids=[scene_index.index_id],
    top_k=10,
)

res.play()

## ⚙️ Understanding and Index Configuration

Configure the understanding run for the visual artifact, then declare how its fields support retrieval.

- `segmentation` defines the timestamped video ranges.
- Analyzer `sampling`, `prompt`, and `schema` define each visual description.
- `use_for` and `fields` define the semantic retrieval contract.
- `callback_url` can notify your application when an understanding run or index completes.

### ⚙️ `segmentation` and `sampling`

Video understanding works over timestamped ranges. This example uses fixed 10-second segments and samples one representative middle frame from each segment for the visual analyzer.

![](https://raw.githubusercontent.com/video-db/videodb-cookbook/main/images/scene_index/VSF.png)

### ⚙️ VLM `prompt` and `schema`

The prompt guides each visual description, and the schema gives the indexed field a stable shape. For activity-focused retrieval, you can use a prompt such as:

> “Describe clearly what is happening in the video. Add `running_detected` if you see a person running.”

Use a matching schema field when you want that value to be searchable.

### ⚙️ `callback_url`

Pass `callback_url` to `video.understand(...)` or `video.index(...)` when your application needs a completion notification.

<div style="height:40px;"></div>

## 🗂️ Managing Indexes
---

>  
> 💡 You can create multiple scene indexes for a video and rank the results after a search before presenting them to your user.

**List indexes created for this video:**

`video.list_indexes()` returns the available index manifests with their names, statuses, and retrieval capabilities.

In [9]:
scene_indexes = video.list_indexes()

for index in scene_indexes:
    print(index.name, index.status, index.use_for)

scene ready ['semantic']


**Inspect one index and preview its records:**

`video.get_index(...)` returns the index manifest and field schema. Use `records()` to inspect a bounded, paginated record preview.

In [10]:
scene_index = video.get_index(index_id=index_id)

print(scene_index)

page = scene_index.records(limit=5)
for record in page.records:
    print(record.start, record.end, record.data)

if page.next_cursor:
    print("More records are available with page.next_cursor.")


Index(index_id=742946ee99054312, video_id=m-z-019f8f57-f740-7d73-86a7-a56cd90c323e, name=scene, status=ready, use_for=['semantic'], record_count=54)
0.0 10.0 {'description': 'A middle-aged man sits in a corporate office, framed in a medium close-up with horizontal blinds and glass partitions behind him. He wears a dark suit, light-colored shirt and striped tie. His expression is serious and slightly weary, with a subtle furrow in his brow and a tilted head suggesting thoughtfulness. Soft, diffused light backlights the scene, creating muted contrasts and a restrained, professional atmosphere. The tidy, minimal background hints at other indistinct figures beyond the glass. The composition emphasizes his authority and introspection, conveying quiet tension and the focused mood of an important conversation or interrogation and strained silence.'}
10.0 20.0 {'description': 'Under a simple outdoor canopy, a crowd stands attentively, many with raised hands and expressive gestures. In front, b

**Delete an index:**

Deleting an index removes its retrieval structures. The original video and stored understanding artifact remain available.

In [11]:
video.delete_index(index_id=index_id)
print("Scene index deleted.")

Scene index deleted.



If you have any questions or feedback, feel free to reach out to us 🙌🏼

* [Discord](https://colab.research.google.com/corgiredirector?site=https%3A%2F%2Fdiscord.gg%2Fpy9P639jGz)
* [GitHub](https://github.com/video-db)
* [Website](https://colab.research.google.com/corgiredirector?site=https%3A%2F%2Fvideodb.io)
* [Email](ashu@videodb.io)